# Toronto Airbnb Market & Performance Benchmarking

**Business question:** How do comparable Toronto Airbnb listings perform, and what
characteristics are associated with stronger estimated performance?

This notebook uses detailed `listings`, `calendar`, and `reviews` data from Inside Airbnb.
The README contains the executive summary and final visuals.

## 1. Setup

Place these files in `../data/` before running:

- `listings.csv.gz`
- `calendar.csv.gz`
- `reviews.csv.gz`

The raw data is intentionally excluded from GitHub.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA = ROOT / "data"
OUTPUTS = ROOT / "outputs"
IMAGES = ROOT / "images"

pd.set_option("display.max_columns", 100)

## 2. Load and clean listings

In [ ]:
listings = pd.read_csv(DATA / "listings.csv.gz", low_memory=False)

listings["price_num"] = pd.to_numeric(
    listings["price"]
    .astype("string")
    .str.replace("$", "", regex=False)
    .str.replace(",", "", regex=False),
    errors="coerce",
)

df = listings[
    listings["price_num"].notna()
    & listings["estimated_revenue_l365d"].notna()
    & (listings["price_num"] > 0)
].copy()

print("Original listings:", f"{len(listings):,}")
print("Listings used for performance analysis:", f"{len(df):,}")

### Why medians matter

The raw listings contain extreme nightly prices. Rather than silently deleting those
observations, the portfolio analysis relies heavily on **medians** and minimum sample
sizes when comparing segments.

## 3. Aggregate the 8M-row calendar file

In [ ]:
valid_ids = set(df["id"].astype("int64"))
monthly_parts = []
listing_parts = []

for chunk in pd.read_csv(
    DATA / "calendar.csv.gz",
    usecols=["listing_id", "date", "available"],
    chunksize=500_000,
):
    chunk = chunk[chunk["listing_id"].isin(valid_ids)].copy()
    chunk["date"] = pd.to_datetime(chunk["date"])
    chunk["unavailable"] = (chunk["available"] == "f").astype("int8")
    chunk["month"] = chunk["date"].dt.to_period("M").astype(str)

    monthly_parts.append(
        chunk.groupby("month")
        .agg(
            calendar_days=("available", "size"),
            unavailable_days=("unavailable", "sum"),
        )
        .reset_index()
    )

    listing_parts.append(
        chunk.groupby("listing_id")
        .agg(
            calendar_days=("available", "size"),
            unavailable_days=("unavailable", "sum"),
        )
        .reset_index()
    )

calendar_monthly = (
    pd.concat(monthly_parts)
    .groupby("month", as_index=False)[["calendar_days", "unavailable_days"]]
    .sum()
)
calendar_monthly["unavailability_rate"] = (
    calendar_monthly["unavailable_days"] / calendar_monthly["calendar_days"]
)

calendar_listing = (
    pd.concat(listing_parts)
    .groupby("listing_id", as_index=False)[["calendar_days", "unavailable_days"]]
    .sum()
)
calendar_listing["calendar_unavailability_rate"] = (
    calendar_listing["unavailable_days"] / calendar_listing["calendar_days"]
)

df = df.merge(
    calendar_listing,
    left_on="id",
    right_on="listing_id",
    how="left",
)

**Interpretation:** `available == "f"` is treated as calendar **unavailability**,
not as a confirmed booking. A host can also block dates manually.

## 4. Market overview by room type

In [ ]:
room_summary = (
    df.groupby("room_type")
    .agg(
        listings=("id", "count"),
        median_price=("price_num", "median"),
        median_estimated_revenue=("estimated_revenue_l365d", "median"),
        median_estimated_occupancy=("estimated_occupancy_l365d", "median"),
        median_reviews_ltm=("number_of_reviews_ltm", "median"),
    )
    .sort_values("listings", ascending=False)
)

room_summary

## 5. Like-for-like neighbourhood benchmark

In [ ]:
one_bed = df[
    (df["room_type"] == "Entire home/apt")
    & (df["bedrooms"] == 1)
].copy()

neighbourhood = (
    one_bed.groupby("neighbourhood_cleansed")
    .agg(
        listings=("id", "count"),
        median_price=("price_num", "median"),
        median_estimated_revenue=("estimated_revenue_l365d", "median"),
        median_estimated_occupancy=("estimated_occupancy_l365d", "median"),
        median_rating=("review_scores_rating", "median"),
        median_reviews_ltm=("number_of_reviews_ltm", "median"),
    )
    .reset_index()
)

benchmark = neighbourhood[
    neighbourhood["listings"] >= 50
].sort_values("median_estimated_revenue", ascending=False)

benchmark.head(10)

Requiring at least 50 listings avoids ranking a neighbourhood based on only a
handful of unusual properties.

## 6. Review seasonality

In [ ]:
reviews = pd.read_csv(
    DATA / "reviews.csv.gz",
    usecols=["listing_id", "date"],
    parse_dates=["date"],
)
reviews = reviews[reviews["listing_id"].isin(valid_ids)].copy()
reviews["year"] = reviews["date"].dt.year
reviews["month_num"] = reviews["date"].dt.month

season = reviews[reviews["year"].isin([2024, 2025])].copy()

counts = (
    season.groupby(["year", "month_num"])
    .size()
    .rename("reviews")
    .reset_index()
)
counts["year_total"] = counts.groupby("year")["reviews"].transform("sum")
counts["monthly_share"] = counts["reviews"] / counts["year_total"]

seasonality = (
    counts.groupby("month_num")
    .agg(avg_monthly_share=("monthly_share", "mean"))
    .reset_index()
)

seasonality

## 7. What distinguishes top-revenue comparable listings?

In [ ]:
q75 = one_bed["estimated_revenue_l365d"].quantile(0.75)

one_bed["performance_group"] = np.where(
    one_bed["estimated_revenue_l365d"] >= q75,
    "Top 25%",
    "Other 75%",
)

performance = (
    one_bed.groupby("performance_group")
    .agg(
        listings=("id", "count"),
        median_estimated_revenue=("estimated_revenue_l365d", "median"),
        median_price=("price_num", "median"),
        median_estimated_occupancy=("estimated_occupancy_l365d", "median"),
        median_rating=("review_scores_rating", "median"),
        median_reviews_ltm=("number_of_reviews_ltm", "median"),
        median_calendar_unavailability=(
            "calendar_unavailability_rate",
            "median",
        ),
        median_minimum_nights=("minimum_nights", "median"),
    )
)

performance["superhost_share"] = (
    one_bed.groupby("performance_group")["host_is_superhost"]
    .apply(lambda s: (s == "t").mean())
)

performance

### Interpretation caution

This is a **descriptive benchmark, not a causal model**. Estimated revenue is itself
constructed from pricing and estimated occupancy, so higher price/occupancy should not
be presented as independent causes of higher modeled revenue.

## 8. Final takeaways

See the repository `README.md` for the polished case study, charts, key findings,
limitations, and reproduction instructions.